In [0]:
%python

#configruacion inicial

from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.sql.types import (
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
)
 
CATALOG = "workspace"
SCHEMA = "bigdata"
 
CSV_INPUT_PATH = (
    "/Volumes/workspace/bigdata/lab_files/lab3/crypto_prices_base_lab_3.csv"
)
 
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.crypto_prices_lab3_bronze"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.crypto_prices_lab3_silver"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.crypto_prices_lab3_gold"
 
TRAIN_RATIO = 0.80
VOLATILITY_PERCENTILE = 0.75
 
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
 
print(f"Fuente CSV: {CSV_INPUT_PATH}")
print(f"Bronze: {BRONZE_TABLE}")
print(f"Silver: {SILVER_TABLE}")
print(f"Gold: {GOLD_TABLE}")

Fuente CSV: /Volumes/workspace/bigdata/lab_files/lab3/crypto_prices_base_lab_3.csv
Bronze: workspace.bigdata.crypto_prices_lab3_bronze
Silver: workspace.bigdata.crypto_prices_lab3_silver
Gold: workspace.bigdata.crypto_prices_lab3_gold


In [0]:
%python

#lectura del database base

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_INPUT_PATH)
)
 
df_raw.printSchema()
display(df_raw.limit(10))
print(f"Total de registros recibidos: {df_raw.count():,}")


root
 |-- coin: string (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- market_cap: double (nullable = true)
 |-- total_volume: double (nullable = true)
 |-- source: string (nullable = true)
 |-- ingestion_date: date (nullable = true)



coin,date,price,market_cap,total_volume,source,ingestion_date
bitcoin,2025-09-10,111521.54423183508,2.221227776260477E12,4.554139428509113E10,coingecko,2026-09-09
bitcoin,2025-09-11,114000.12092296442,2.2707858506830293E12,5.204769660798891E10,coingecko,2026-09-09
bitcoin,2025-09-12,115553.48783920688,2.303507550746883E12,4.457346085540361E10,coingecko,2026-09-09
bitcoin,2025-09-13,116091.80704192116,2.3128787950347817E12,5.076551346376378E10,coingecko,2026-09-09
bitcoin,2025-09-14,115974.8873939637,2.3101869957799624E12,2.9926419199882915E10,coingecko,2026-09-09
bitcoin,2025-09-15,115278.5525158622,2.298366422491926E12,2.7220055788518738E10,coingecko,2026-09-09
bitcoin,2025-09-16,115356.68489564986,2.298376374765022E12,4.6807932859599625E10,coingecko,2026-09-09
bitcoin,2025-09-17,116797.00205991932,2.3262521110150996E12,4.011565562262766E10,coingecko,2026-09-09
bitcoin,2025-09-18,116430.93170207892,2.31986226330109E12,5.507809955769372E10,coingecko,2026-09-09
bitcoin,2025-09-19,117169.11793707692,2.3341552703816797E12,4.3746882355192566E10,coingecko,2026-09-09


Total de registros recibidos: 2,920


In [0]:
%python
#Capa bronze - conserva el dato recibido con tranformaciones minimas y agrega campos de control o trazabilidad

df_bronze = (
    df_raw
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("record_source", F.lit("coingecko_dataset"))
)
 
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)
 
print(f"Tabla Bronze creada: {BRONZE_TABLE}")
display(spark.table(BRONZE_TABLE).limit(10))

Tabla Bronze creada: workspace.bigdata.crypto_prices_lab3_bronze


coin,date,price,market_cap,total_volume,source,ingestion_date,ingestion_timestamp,record_source
bitcoin,2025-09-10,111521.54423183508,2.221227776260477E12,4.554139428509113E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset
bitcoin,2025-09-11,114000.12092296442,2.2707858506830293E12,5.204769660798891E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset
bitcoin,2025-09-12,115553.48783920688,2.303507550746883E12,4.457346085540361E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset
bitcoin,2025-09-13,116091.80704192116,2.3128787950347817E12,5.076551346376378E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset
bitcoin,2025-09-14,115974.8873939637,2.3101869957799624E12,2.9926419199882915E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset
bitcoin,2025-09-15,115278.5525158622,2.298366422491926E12,2.7220055788518738E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset
bitcoin,2025-09-16,115356.68489564986,2.298376374765022E12,4.6807932859599625E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset
bitcoin,2025-09-17,116797.00205991932,2.3262521110150996E12,4.011565562262766E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset
bitcoin,2025-09-18,116430.93170207892,2.31986226330109E12,5.507809955769372E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset
bitcoin,2025-09-19,117169.11793707692,2.3341552703816797E12,4.3746882355192566E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset


In [0]:
%python
'''Capa Silver 
tipamos columnas
eliminanos registros invalidos
eliminamos duplicados
generamos varibles temporales basicas
'''

df_silver = (
    spark.table(BRONZE_TABLE)
    .withColumn("coin", F.trim(F.col("coin")))
    .withColumn("date", F.col("date").cast("date"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("market_cap", F.col("market_cap").cast("double"))
    .withColumn("total_volume", F.col("total_volume").cast("double"))
    .withColumn("ingestion_date", F.col("ingestion_date").cast("date"))
    .filter(F.col("coin").isNotNull())
    .filter(F.col("date").isNotNull())
    .filter(F.col("price") > 0)
    .filter(F.col("market_cap") > 0)
    .filter(F.col("total_volume") >= 0)
    .dropDuplicates(["coin", "date"])
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
)
 
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)
 
print(f"Tabla Silver creada: {SILVER_TABLE}")


Tabla Silver creada: workspace.bigdata.crypto_prices_lab3_silver


In [0]:
%python
#Validacion de cobertura temporal por criptomoneda
display(
    df_silver
    .groupBy("coin")
    .agg(
        F.count("*").alias("records"),
        F.min("date").alias("first_date"),
        F.max("date").alias("last_date"),
        F.round(F.avg("price"), 4).alias("avg_price"),
    )
    .orderBy("avg_price")
)

coin,records,first_date,last_date,avg_price
tron,365,2025-09-10,2026-09-09,0.3144
cardano,365,2025-09-10,2026-09-09,0.3553
polkadot,365,2025-09-10,2026-09-09,1.7783
chainlink,365,2025-09-10,2026-09-09,11.9506
litecoin,365,2025-09-10,2026-09-09,66.7096
solana,365,2025-09-10,2026-09-09,115.0823
ethereum,365,2025-09-10,2026-09-09,2623.3646
bitcoin,365,2025-09-10,2026-09-09,81796.7119


In [0]:
%python
#Capa gold featuring engineering temporal
# contruir caracteristicas usando unicamente informacion disponible hasta cada fecha
 
df = spark.table(SILVER_TABLE)
w_order = Window.partitionBy("coin").orderBy("date")
 
df_features = (
    df
    .withColumn("lag_price_1", F.lag("price", 1).over(w_order))
    .withColumn("lag_price_3", F.lag("price", 3).over(w_order))
    .withColumn("lag_price_7", F.lag("price", 7).over(w_order))
    .withColumn("lag_volume_1", F.lag("total_volume", 1).over(w_order))
    .withColumn("lag_market_cap_1", F.lag("market_cap", 1).over(w_order))
)
 

In [0]:
%python
#Retornos

def safe_divide(numerator, denominator):
    """Divide dos columnas evitando divisiones por cero."""
    return F.when(
        denominator.isNotNull() & (denominator != 0),
        numerator / denominator,
    )
 
 
df_features = (
    df_features
    .withColumn(
        "return_1d",
        safe_divide(F.col("price") - F.col("lag_price_1"), F.col("lag_price_1")),
    )
    .withColumn(
        "return_3d",
        safe_divide(F.col("price") - F.col("lag_price_3"), F.col("lag_price_3")),
    )
    .withColumn(
        "return_7d",
        safe_divide(F.col("price") - F.col("lag_price_7"), F.col("lag_price_7")),
    )
)

In [0]:
%python
display(df_features.head(10))
       

coin,date,price,market_cap,total_volume,source,ingestion_date,ingestion_timestamp,record_source,year,month,day_of_week,lag_price_1,lag_price_3,lag_price_7,lag_volume_1,lag_market_cap_1,return_1d,return_3d,return_7d
bitcoin,2025-09-10,111521.54423183508,2.221227776260477E12,4.554139428509113E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset,2025,9,4,null,null,null,null,null,null,null,null
bitcoin,2025-09-11,114000.12092296442,2.2707858506830293E12,5.204769660798891E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset,2025,9,5,111521.54423183508,null,null,4.554139428509113E10,2.221227776260477E12,0.022225092991689354,null,null
bitcoin,2025-09-12,115553.48783920688,2.303507550746883E12,4.457346085540361E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset,2025,9,6,114000.12092296442,null,null,5.204769660798891E10,2.2707858506830293E12,0.013626011127586003,null,null
bitcoin,2025-09-13,116091.80704192116,2.3128787950347817E12,5.076551346376378E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset,2025,9,7,115553.48783920688,111521.54423183508,null,4.457346085540361E10,2.303507550746883E12,0.004658614921804461,0.04098098570609141,null
bitcoin,2025-09-14,115974.8873939637,2.3101869957799624E12,2.9926419199882915E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset,2025,9,1,116091.80704192116,114000.12092296442,null,5.076551346376378E10,2.3128787950347817E12,-0.001007130915924499,0.017322494529051692,null
bitcoin,2025-09-15,115278.5525158622,2.298366422491926E12,2.7220055788518738E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset,2025,9,2,115974.8873939637,115553.48783920688,null,2.9926419199882915E10,2.3101869957799624E12,-0.00600418671445709,-0.002379290564792368,null
bitcoin,2025-09-16,115356.68489564986,2.298376374765022E12,4.6807932859599625E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset,2025,9,3,115278.5525158622,116091.80704192116,null,2.7220055788518738E10,2.298366422491926E12,6.77770305772212E-4,-0.006332248286960035,null
bitcoin,2025-09-17,116797.00205991932,2.3262521110150996E12,4.011565562262766E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset,2025,9,4,115356.68489564986,115974.8873939637,111521.54423183508,4.6807932859599625E10,2.298376374765022E12,0.012485771115669259,0.0070887300210339755,0.04730438288334157
bitcoin,2025-09-18,116430.93170207892,2.31986226330109E12,5.507809955769372E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset,2025,9,5,116797.00205991932,115278.5525158622,114000.12092296442,4.011565562262766E10,2.3262521110150996E12,-0.0031342444701842995,0.009996475155759322,0.021322878953410247
bitcoin,2025-09-19,117169.11793707692,2.3341552703816797E12,4.3746882355192566E10,coingecko,2026-09-09,2026-09-17T22:38:24.445Z,coingecko_dataset,2025,9,6,116430.93170207892,115356.68489564986,115553.48783920688,5.507809955769372E10,2.31986226330109E12,0.006340121342384026,0.01571155623158354,0.013981664492189115


In [0]:
%python
#retornos
#calcula la diferencia de un precio del pasado

def safe_divide(numerator, denominator):
    """Divide dos columnas evitando divisiones por cero."""
    return F.when(
        denominator.isNotNull() & (denominator != 0),
        numerator / denominator,
    )
 
 
df_features = (
    df_features
    .withColumn(
        "return_1d",
        safe_divide(F.col("price") - F.col("lag_price_1"), F.col("lag_price_1")),
    )
    .withColumn(
        "return_3d",
        safe_divide(F.col("price") - F.col("lag_price_3"), F.col("lag_price_3")),
    )
    .withColumn(
        "return_7d",
        safe_divide(F.col("price") - F.col("lag_price_7"), F.col("lag_price_7")),
    )
)

In [0]:
%python
#medias moviles y volatilidad historica
w3 = Window.partitionBy("coin").orderBy("date").rowsBetween(-2, 0)
w7 = Window.partitionBy("coin").orderBy("date").rowsBetween(-6, 0)
w14 = Window.partitionBy("coin").orderBy("date").rowsBetween(-13, 0)
 
df_features = (
    df_features
    .withColumn("ma_3", F.avg("price").over(w3))
    .withColumn("ma_7", F.avg("price").over(w7))
    .withColumn("ma_14", F.avg("price").over(w14))
    .withColumn("volatility_3d", F.stddev("return_1d").over(w3))
    .withColumn("volatility_7d", F.stddev("return_1d").over(w7))
    .withColumn("volatility_14d", F.stddev("return_1d").over(w14))
    .withColumn("volume_ma_7", F.avg("total_volume").over(w7))
)

In [0]:
%python
#variables relativas del mercado
df_features = (
    df_features
    .withColumn(
        "volume_change_1d",
        safe_divide(
            F.col("total_volume") - F.col("lag_volume_1"),
            F.col("lag_volume_1"),
        ),
    )
    .withColumn(
        "market_cap_change_1d",
        safe_divide(
            F.col("market_cap") - F.col("lag_market_cap_1"),
            F.col("lag_market_cap_1"),
        ),
    )
    .withColumn("volume_vs_ma_7", safe_divide(F.col("total_volume"), F.col("volume_ma_7")))
    .withColumn("price_vs_ma_7", safe_divide(F.col("price"), F.col("ma_7")))
    .withColumn("price_vs_ma_14", safe_divide(F.col("price"), F.col("ma_14")))
)

In [0]:
%python
#contruccion comportamiento futuro
#next_price y #abs_return... se utilizan exclusivamente para construir el target. No son features


df_features = (
    df_features
    .withColumn("next_date", F.lead("date", 1).over(w_order))
    .withColumn("next_price", F.lead("price", 1).over(w_order))
    .withColumn(
        "return_next_day",
        safe_divide(F.col("next_price") - F.col("price"), F.col("price")),
    )
    .withColumn("abs_return_next_day", F.abs(F.col("return_next_day")))
)
 
# Garantiza que la siguiente observación corresponda al día calendario siguiente.
df_features = df_features.filter(
    F.datediff(F.col("next_date"), F.col("date")) == 1
)

In [0]:
%python
#Seleccion de variables de modelado
feature_cols = [
    "month",
    "day_of_week",
    "return_1d",
    "return_3d",
    "return_7d",
    "volatility_3d",
    "volatility_7d",
    "volatility_14d",
    "volume_change_1d",
    "market_cap_change_1d",
    "volume_vs_ma_7",
    "price_vs_ma_7",
    "price_vs_ma_14",
]
df_gold = df_features
for column in feature_cols:
    df_gold = df_gold.filter(F.col(column).isNotNull())
df_gold = df_gold.filter(F.col("abs_return_next_day").isNotNull())
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE)
)
print(f"Tabla Gold creada: {GOLD_TABLE}")
print(f"Registros Gold: {df_gold.count():,}")

Tabla Gold creada: workspace.bigdata.crypto_prices_lab3_gold
Registros Gold: 2,856


In [0]:
%python
#exploracion capa gold

display(
    spark.sql(
        f"""
        SELECT
            coin,
            COUNT(*) AS records,
            ROUND(AVG(ABS(return_1d)), 6) AS avg_abs_return_1d,
            ROUND(AVG(volatility_7d), 6) AS avg_volatility_7d,
            ROUND(AVG(volume_vs_ma_7), 4) AS avg_volume_vs_ma_7
        FROM {GOLD_TABLE}
        GROUP BY coin
        ORDER BY avg_volatility_7d DESC
        """
    )
)

coin,records,avg_abs_return_1d,avg_volatility_7d,avg_volume_vs_ma_7
cardano,357,0.031295,0.03854,0.9985
polkadot,357,0.028879,0.037342,1.0058
chainlink,357,0.028053,0.035237,1.0022
solana,357,0.025977,0.032181,0.9971
ethereum,357,0.02331,0.030177,0.9964
litecoin,357,0.020855,0.027392,1.0023
bitcoin,357,0.016398,0.020438,1.0004
tron,357,0.009682,0.012381,0.9978


Databricks visualization. Run in Databricks to view.

In [0]:
%python

display(
    spark.sql(
        f"""
        SELECT
            coin,
            date,
            price,
            return_1d,
            return_7d,
            volatility_7d,
            volume_vs_ma_7
        FROM {GOLD_TABLE}
        WHERE coin = 'bitcoin'
        ORDER BY date
        """
    )
)

coin,date,price,return_1d,return_7d,volatility_7d,volume_vs_ma_7
bitcoin,2025-09-17,116797.00205991932,0.012485771115669259,0.04730438288334157,0.009863196951407995,0.9634692090358076
bitcoin,2025-09-18,116430.93170207892,-0.0031342444701842995,0.021322878953410247,0.007591487235825033,1.3092140470940747
bitcoin,2025-09-19,117169.11793707692,0.006340121342384026,0.013981664492189115,0.006285894811570216,1.0427964098699574
bitcoin,2025-09-20,115656.94582205842,-0.012905893136710157,-0.003745839012616179,0.008248262540670572,0.8617924447183507
bitcoin,2025-09-21,115722.89635080755,5.70225404798434E-4,-0.002172806965529846,0.008253860843480627,0.48488086943691594
bitcoin,2025-09-22,115255.84820116172,-0.004035918252771753,-1.9695176773982997E-4,0.008057608724217838,0.51277245887336
bitcoin,2025-09-23,112692.49699542264,-0.022240530487139646,-0.02309521899521651,0.011609496673308737,1.737657248143669
bitcoin,2025-09-24,112002.46136592983,-0.006123172774500098,-0.041050203424997066,0.009300056939170576,1.129643861429023
bitcoin,2025-09-25,113317.1626894919,0.011738146711497087,-0.02674348617731981,0.011487512882063516,1.1751866460649356
bitcoin,2025-09-26,109233.05041611978,-0.0360414272334305,-0.0677317339302583,0.01565873255906658,1.6961375984284364


In [0]:
%python
#division temporal train/test
#en una serie temporal el pasado entrena y el futuro evalua

df_ml = spark.table(GOLD_TABLE)
distinct_dates = df_ml.select("date").distinct()
n_dates = distinct_dates.count()
 
split_position = max(2, int(n_dates * TRAIN_RATIO))
date_rank_window = Window.orderBy("date")
 
split_date_row = (
    distinct_dates
    .withColumn("_date_rank", F.row_number().over(date_rank_window))
    .filter(F.col("_date_rank") == split_position + 1)
    .select("date")
    .first()
)
 
if split_date_row is None:
    raise ValueError("No fue posible determinar la fecha de corte temporal.")
 
split_date = split_date_row["date"]
 
print(f"Fechas distintas: {n_dates}")
print(f"Fecha de corte temporal: {split_date}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Fechas distintas: 357
Fecha de corte temporal: 2026-06-29


In [0]:
%python
#para evitar contaminacion de la frontera temporal, una fila de entrenamiento solo se conserva si su next_date tambien pertenece al periodo de entrenamiento 

train_base = df_ml.filter(
    (F.col("date") < F.lit(split_date))
    & (F.col("next_date") < F.lit(split_date))
)
 
test_base = df_ml.filter(F.col("date") >= F.lit(split_date))
 
print(f"Train: {train_base.count():,} registros")
print(f"Test:  {test_base.count():,} registros")

Train: 2,272 registros
Test:  576 registros


In [0]:
%python
#Contruccion del target sin leakage
#el umbral de alta volatilidad se calcula exclusivamente con Train - 75%

thresholds_df = (
    train_base
    .groupBy("coin")
    .agg(
        F.expr(
            f"percentile_approx(abs_return_next_day, {VOLATILITY_PERCENTILE}, 10000)"
        ).alias("volatility_threshold")
    )
)
 
display(thresholds_df.orderBy("coin"))


coin,volatility_threshold
bitcoin,0.022921171539970153
cardano,0.041745861765515556
chainlink,0.03765322033385321
ethereum,0.033134996893382486
litecoin,0.028814187328833042
polkadot,0.03686203119571053
solana,0.03948490731535609
tron,0.014101057708362555


In [0]:
%python
def add_target(df):
    """Aplica los umbrales calculados exclusivamente en train."""
    return (
        df
        .join(thresholds_df, on="coin", how="inner")
        .withColumn(
            "target_high_volatility_next_day",
            F.when(
                F.col("abs_return_next_day") >= F.col("volatility_threshold"),
                F.lit(1.0),
            ).otherwise(F.lit(0.0)),
        )
    )
 
 
train_df = add_target(train_base)
test_df = add_target(test_base)

In [0]:
%python
#distribucion de la variable objetivo

display(
    train_df
    .groupBy("target_high_volatility_next_day")
    .count()
    .orderBy("target_high_volatility_next_day")
)
 
display(
    test_df
    .groupBy("target_high_volatility_next_day")
    .count()
    .orderBy("target_high_volatility_next_day")
)

target_high_volatility_next_day,count
0.0,1696
1.0,576


target_high_volatility_next_day,count
0.0,481
1.0,95


In [0]:
%python
#Baseline
#predecir siempre que no habra volatilidad

baseline_predictions = test_df.withColumn("prediction", F.lit(0.0))

In [0]:
%python
#preparacion para spark ml
#coin se indexa y luego se transforma mediante one-hot encoding para evitar imponer un order artificial entre criptomonedas

coin_indexer = StringIndexer(
    inputCol="coin",
    outputCol="coin_index",
    handleInvalid="keep",
)
 
coin_encoder = OneHotEncoder(
    inputCols=["coin_index"],
    outputCols=["coin_ohe"],
    handleInvalid="keep",
)
 
assembler = VectorAssembler(
    inputCols=["coin_ohe"] + feature_cols,
    outputCol="features",
    handleInvalid="skip",
)

In [0]:
%python
#peso de la clase positiva

class_counts = {
    float(row["target_high_volatility_next_day"]): row["count"]
    for row in (
        train_df
        .groupBy("target_high_volatility_next_day")
        .count()
        .collect()
    )
}
 
n0 = class_counts.get(0.0, 1)
n1 = class_counts.get(1.0, 1)
positive_weight = n0 / n1
 
print(f"Clase 0: {n0:,}")
print(f"Clase 1: {n1:,}")
print(f"Peso de la clase positiva: {positive_weight:.3f}")
 
train_weighted_df = train_df.withColumn(
    "class_weight",
    F.when(
        F.col("target_high_volatility_next_day") == 1.0,
        F.lit(float(positive_weight)),
    ).otherwise(F.lit(1.0)),
)

Clase 0: 1,696
Clase 1: 576
Peso de la clase positiva: 2.944


In [0]:
%python
#modelo 1 logistic regression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="target_high_volatility_next_day",
    predictionCol="prediction",
    rawPredictionCol="rawPrediction",
    probabilityCol="probability",
    weightCol="class_weight",
    maxIter=100,
)
 
lr_pipeline = Pipeline(stages=[coin_indexer, coin_encoder, assembler, lr])
lr_model = lr_pipeline.fit(train_weighted_df)
lr_predictions = lr_model.transform(test_df)
 
display(
    lr_predictions.select(
        "coin",
        "date",
        "target_high_volatility_next_day",
        "prediction",
        "probability",
    ).limit(20)
)

coin,date,target_high_volatility_next_day,prediction,probability
bitcoin,2026-06-29,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5142567594872964"",""0.48574324051270357""]}"
bitcoin,2026-06-30,1.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.50686514715412"",""0.49313485284588""]}"
bitcoin,2026-07-01,1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.4974173901311356"",""0.5025826098688644""]}"
bitcoin,2026-07-02,1.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5339916828483507"",""0.46600831715164925""]}"
bitcoin,2026-07-03,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5846447870874797"",""0.4153552129125203""]}"
bitcoin,2026-07-04,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6141893492025438"",""0.38581065079745624""]}"
bitcoin,2026-07-05,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.563568905334563"",""0.43643109466543695""]}"
bitcoin,2026-07-06,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6001665332282005"",""0.39983346677179954""]}"
bitcoin,2026-07-07,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5354367541699302"",""0.4645632458300698""]}"
bitcoin,2026-07-08,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5794685966830672"",""0.42053140331693284""]}"


In [0]:
%python
#modelo 2 - random forest

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="target_high_volatility_next_day",
    predictionCol="prediction",
    rawPredictionCol="rawPrediction",
    probabilityCol="probability",
    weightCol="class_weight",
    numTrees=100,
    maxDepth=6,
    seed=42,
)
 
rf_pipeline = Pipeline(stages=[coin_indexer, coin_encoder, assembler, rf])
rf_model = rf_pipeline.fit(train_weighted_df)
rf_predictions = rf_model.transform(test_df)
 
display(
    rf_predictions.select(
        "coin",
        "date",
        "target_high_volatility_next_day",
        "prediction",
        "probability",
    ).limit(20)
)

coin,date,target_high_volatility_next_day,prediction,probability
bitcoin,2026-06-29,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6357063620461518"",""0.3642936379538482""]}"
bitcoin,2026-06-30,1.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6384005230048604"",""0.3615994769951397""]}"
bitcoin,2026-07-01,1.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5844140610406711"",""0.41558593895932894""]}"
bitcoin,2026-07-02,1.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6246573575873637"",""0.37534264241263626""]}"
bitcoin,2026-07-03,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6075581247506764"",""0.39244187524932356""]}"
bitcoin,2026-07-04,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7579497410319062"",""0.24205025896809382""]}"
bitcoin,2026-07-05,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6002146014760898"",""0.3997853985239101""]}"
bitcoin,2026-07-06,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5673220103764751"",""0.43267798962352494""]}"
bitcoin,2026-07-07,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.575075120730887"",""0.424924879269113""]}"
bitcoin,2026-07-08,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5637621273690339"",""0.4362378726309661""]}"


In [0]:
%python
#evaluacion
#evaluacion la clase positiva 1 = alta volatilidad mediante accurancy, precision, recall, F1, ROC AUC y PR AUC
roc_evaluator = BinaryClassificationEvaluator(
    labelCol="target_high_volatility_next_day",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)
pr_evaluator = BinaryClassificationEvaluator(
    labelCol="target_high_volatility_next_day",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR",
)
def confusion_metrics(predictions):
    """Calcula métricas binarias enfocadas en la clase positiva."""
    label = "target_high_volatility_next_day"
    row = predictions.agg(
        F.sum(
            F.when(
                (F.col(label) == 1.0) & (F.col("prediction") == 1.0),
                1,
            ).otherwise(0)
        ).alias("tp"),
        F.sum(
            F.when(
                (F.col(label) == 0.0) & (F.col("prediction") == 1.0),
                1,
            ).otherwise(0)
        ).alias("fp"),
        F.sum(
            F.when(
                (F.col(label) == 1.0) & (F.col("prediction") == 0.0),
                1,
            ).otherwise(0)
        ).alias("fn"),
        F.sum(
            F.when(
                (F.col(label) == 0.0) & (F.col("prediction") == 0.0),
                1,
            ).otherwise(0)
        ).alias("tn"),
    ).first()
 
    tp = int(row["tp"] or 0)
    fp = int(row["fp"] or 0)
    fn = int(row["fn"] or 0)
    tn = int(row["tn"] or 0)
    total = tp + fp + fn + tn
 
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0.0
    )
 
    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }
 
 
def evaluate_model(predictions, model_name):
    metrics = confusion_metrics(predictions)
    return (
        model_name,
        metrics["accuracy"],
        metrics["precision"],
        metrics["recall"],
        metrics["f1"],
        float(roc_evaluator.evaluate(predictions)),
        float(pr_evaluator.evaluate(predictions)),
        metrics["tp"],
        metrics["fp"],
        metrics["fn"],
        metrics["tn"],
    )


In [0]:
%python

baseline_metrics = confusion_metrics(baseline_predictions)
 
baseline_result = (
    "Baseline_always_0",
    baseline_metrics["accuracy"],
    baseline_metrics["precision"],
    baseline_metrics["recall"],
    baseline_metrics["f1"],
    None,
    None,
    baseline_metrics["tp"],
    baseline_metrics["fp"],
    baseline_metrics["fn"],
    baseline_metrics["tn"],
)

In [0]:
%python
#comparacion de modelos

results = [
    baseline_result,
    evaluate_model(lr_predictions, "LogisticRegression"),
    evaluate_model(rf_predictions, "RandomForest"),
]
 
metrics_schema = StructType(
    [
        StructField("model", StringType(), False),
        StructField("accuracy", DoubleType(), True),
        StructField("precision", DoubleType(), True),
        StructField("recall", DoubleType(), True),
        StructField("f1", DoubleType(), True),
        StructField("roc_auc", DoubleType(), True),
        StructField("pr_auc", DoubleType(), True),
        StructField("tp", LongType(), True),
        StructField("fp", LongType(), True),
        StructField("fn", LongType(), True),
        StructField("tn", LongType(), True),
    ]
)
 
metrics_df = spark.createDataFrame(results, schema=metrics_schema)
display(metrics_df.orderBy(F.desc("f1")))

model,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,fn,tn
LogisticRegression,0.7378472222222222,0.18888888888888888,0.17894736842105263,0.1837837837837838,0.49434292592187296,0.17610560096065242,17,73,78,408
RandomForest,0.8177083333333334,0.2222222222222222,0.042105263157894736,0.07079646017699114,0.5353102089944202,0.16943495147028448,4,14,91,467
Baseline_always_0,0.8350694444444444,0.0,0.0,0.0,null,null,0,0,95,481


In [0]:
%python
#matriz de confusion - logistic

display(
    lr_predictions
    .groupBy("target_high_volatility_next_day", "prediction")
    .count()
    .orderBy("target_high_volatility_next_day", "prediction")
)

target_high_volatility_next_day,prediction,count
0.0,0.0,408
0.0,1.0,73
1.0,0.0,78
1.0,1.0,17


In [0]:
%python
#matriz de confusion - ramdom

display(
    rf_predictions
    .groupBy("target_high_volatility_next_day", "prediction")
    .count()
    .orderBy("target_high_volatility_next_day", "prediction")
)

target_high_volatility_next_day,prediction,count
0.0,0.0,467
0.0,1.0,14
1.0,0.0,91
1.0,1.0,4


In [0]:
%python
#Importancia de variables

def get_vector_feature_names(df, vector_col="features"):
    """Extrae los nombres de atributos desde el metadata del vector Spark ML."""
    attrs = (
        df.schema[vector_col]
        .metadata
        .get("ml_attr", {})
        .get("attrs", {})
    )
 
    indexed_names = {}
    for attr_group in attrs.values():
        for attr in attr_group:
            indexed_names[int(attr["idx"])] = attr["name"]
 
    if not indexed_names:
        return []
 
    return [
        indexed_names.get(i, f"feature_{i}")
        for i in range(max(indexed_names) + 1)
    ]
 
 
rf_classifier = rf_model.stages[-1]
importance_values = rf_classifier.featureImportances.toArray().tolist()
rf_feature_names = get_vector_feature_names(rf_predictions)
 
if len(rf_feature_names) != len(importance_values):
    rf_feature_names = [f"feature_{i}" for i in range(len(importance_values))]
 
feature_importance_rows = [
    (name, float(importance))
    for name, importance in zip(rf_feature_names, importance_values)
]
 
feature_importance_df = spark.createDataFrame(
    feature_importance_rows,
    ["feature", "importance"],
)
 
display(feature_importance_df.orderBy(F.desc("importance")))



feature,importance
return_7d,0.10019141199396833
day_of_week,0.0992091021708846
price_vs_ma_7,0.08542463660204232
month,0.08259173532419396
volume_change_1d,0.07978950687698623
return_3d,0.07441686725393609
price_vs_ma_14,0.07232246502953155
volatility_3d,0.072018922506152
volume_vs_ma_7,0.07032323529505642
volatility_7d,0.06415583667547975


In [0]:
%python
#analisis adicinal por moneda

display(
    rf_predictions
    .groupBy("coin")
    .agg(
        F.count("*").alias("records"),
        F.round(
            F.avg("target_high_volatility_next_day"),
            4,
        ).alias("actual_high_volatility_rate"),
        F.round(
            F.avg("prediction"),
            4,
        ).alias("predicted_high_volatility_rate"),
    )
    .orderBy("coin")
)

coin,records,actual_high_volatility_rate,predicted_high_volatility_rate
bitcoin,72,0.1806,0.0
cardano,72,0.2639,0.0694
chainlink,72,0.1806,0.0417
ethereum,72,0.125,0.0417
litecoin,72,0.1389,0.0139
polkadot,72,0.1944,0.0556
solana,72,0.1111,0.0278
tron,72,0.125,0.0


In [0]:
%python
from pyspark.ml.functions import vector_to_array
 
rf_probabilities = (
    rf_predictions
    .withColumn(
        "prob_high_volatility",
        vector_to_array("probability")[1]
    )
)
 
display(
    rf_probabilities.select(
        "coin",
        "date",
        "target_high_volatility_next_day",
        "prob_high_volatility",
        "prediction",
    )
    .orderBy(F.desc("prob_high_volatility"))
)

coin,date,target_high_volatility_next_day,prob_high_volatility,prediction
polkadot,2026-06-29,0.0,0.6120846886781401,1.0
polkadot,2026-07-29,0.0,0.5984593843277962,1.0
polkadot,2026-07-30,0.0,0.5727582633689189,1.0
cardano,2026-07-28,0.0,0.5653100322066891,1.0
cardano,2026-07-06,0.0,0.5529174999780627,1.0
cardano,2026-09-08,0.0,0.5234756281910242,1.0
solana,2026-08-26,1.0,0.5195301954916003,1.0
chainlink,2026-08-24,0.0,0.5190657066867255,1.0
chainlink,2026-09-08,1.0,0.5182165551772245,1.0
ethereum,2026-08-23,0.0,0.517648884193543,1.0


In [0]:
%python
display(
    rf_probabilities.select(
        "prob_high_volatility"
    )
)

prob_high_volatility
0.3642936379538482
0.3615994769951397
0.41558593895932894
0.37534264241263626
0.39244187524932356
0.24205025896809382
0.3997853985239101
0.43267798962352494
0.424924879269113
0.4362378726309661
